# Laboratorio 3 — Preprocesamiento de Datos
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 19 de agosto de 2026 · Sesión 3 de 17
**Bloques:** 6-8 (11:40-13:40) · Lab. Central 114
**Unidad 1:** Fundamentos y Preparación

**Objetivo:** cerrar en código lo visto en la teoría. Primero, medir el **impacto real del
escalamiento** en Wine, con una comparación más justa que la de las clases anteriores. Después,
enfrentar un dataset sucio de verdad — **Titanic**, con nulos y columnas de texto — y recorrer el
flujo completo: nulos → categóricas → escalamiento → modelo.

> **De dónde venimos:** en el Lab 1, KNN quedó en ~0.79 mientras el árbol sacó 1.0; en el mini-lab
> de la clase 2, K-means agrupó mediocre. En los tres casos el villano fue el mismo: **las escalas
> dispares** que viste en `describe()`. Hoy ese villano cae.

> **Cómo usar este notebook:** las celdas de carga y de split vienen dadas (el porqué del split es
> la clase 5). El resto del código **lo escribes tú**, con instrucciones en cada sección.
> Al terminar: `Kernel → Restart & Run All` y revisar que corra de arriba a abajo sin errores.

---
## Parte 1 · El impacto real del escalamiento en Wine

### 1. Setup y split (código dado)

Mismo Wine de siempre, con una novedad: apartamos un 30% de los datos que el modelo **no verá
durante el entrenamiento**, y el score se medirá solo sobre esa parte.

Hoy usamos `train_test_split` como herramienta, sin justificarlo del todo — el porqué exacto es la
**clase 5** completa. Por ahora basta la intuición: medir sobre los mismos datos con que se entrenó
es tomar la prueba con las respuestas a la vista (el 1.0 del árbol en el Lab 1…).

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("train:", X_train.shape, "| test:", X_test.shape)

### 2. KNN sin escalar

El mismo KNN del Lab 1, pero ahora entrenado y evaluado como corresponde: `fit` con los datos de
**train**, `score` con los de **test**.

**Tu tarea:**
1. Crea un `KNeighborsClassifier` (parámetros por defecto).
2. Entrénalo con los datos de entrenamiento.
3. Imprime su `score` sobre los datos de test.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier



**Responde aquí (doble clic para editar):** en el Lab 1, KNN evaluado sobre sus propios datos de
entrenamiento daba ~0.79. ¿El score de ahora es mejor o peor? ¿Por qué tiene sentido que así sea?

> *(tu respuesta)*

### 3. KNN con `StandardScaler` — la regla de oro

Ahora escalamos las features **antes** de entrenar. La regla de oro de la teoría, en código:

- `fit_transform()` **solo con train** → las medias y desviaciones se aprenden del entrenamiento.
- `transform()` (sin `fit`) **con test** → los datos nuevos se transforman con los estadísticos de train.

**Tu tarea:**
1. Crea un `StandardScaler`.
2. Obtén `X_train_scaled` con `fit_transform` sobre `X_train`.
3. Obtén `X_test_scaled` con `transform` sobre `X_test`.
4. Entrena un KNN nuevo con los datos escalados e imprime su score sobre el test escalado.

In [ ]:
from sklearn.preprocessing import StandardScaler



**Antes de seguir, responde:** ¿qué pasaría si en el paso 3 hubieras usado
`scaler.fit_transform(X_test)`? El código correría igual, y hasta daría un número parecido…
¿por qué estaría **mal** de todas formas? *(Pista: en producción los vinos nuevos llegan de a uno.)*

> *(tu respuesta)*

### 4. Discusión

Compara los dos scores lado a lado y responde:

1. ¿Cuánto mejoró KNN? Ojo: **no tocamos el algoritmo** — solo cambiamos cómo le llegan los datos.
2. ¿Qué feature dominaba la distancia euclidiana antes de escalar? ¿Qué pasó con su influencia
   después? *(Si dudas, vuelve al `describe()` del Lab 1.)*
3. ¿Cambiaría el score de un **árbol de decisión** al escalar? ¿Por qué? — responde primero, y
   si quieres compruébalo en la celda de abajo.

> *(tus respuestas)*

In [ ]:
# Comprobación opcional: árbol de decisión con y sin escalar (random_state=42)
from sklearn.tree import DecisionTreeClassifier



---
## ☕ Recreo
---
## Parte 2 · Un dataset sucio de verdad: Titanic

### 5. Cargar y diagnosticar el desastre

891 pasajeros del Titanic, con lo que los datos reales siempre traen: **valores faltantes** y
**columnas de texto**. Wine, curado en laboratorio, no tenía nada de esto.

La celda de carga viene dada, con respaldo local por si falla internet.

In [ ]:
import pandas as pd

try:
    import seaborn as sns
    titanic = sns.load_dataset("titanic")   # necesita internet la primera vez
except Exception:
    titanic = pd.read_csv("titanic.csv")    # respaldo local, misma carpeta del notebook

print(titanic.shape)
titanic.head()

**Tu tarea — el diagnóstico** (lo mismo que hiciste con Wine en el Lab 1, una celda por cosa):
1. `info()` general.
2. Nulos por columna.
3. Tipos de datos: ¿qué columnas son texto?

**Responde aquí:** ¿qué columnas tienen nulos y cuántos? ¿Cuáles vienen como texto? ¿Alguna
columna parece **duplicada** de otra? *(mira `embarked` / `embark_town`, `pclass` / `class`,
`survived` / `alive`…)*

> *(tus respuestas)*

### 6. Imputación de nulos

Tres columnas con nulos, tres decisiones distintas. **Antes de escribir código**, lee la tabla y
piensa por qué cada estrategia es distinta:

| Columna | Nulos | Estrategia |
|---|---|---|
| `age` | 177 de 891 | imputar con la **mediana** |
| `embarked` | 2 de 891 | imputar con la **moda** |
| `deck` | 688 de 891 | **eliminar la columna** completa |

**Tu tarea:** aplica las tres y verifica que no queden nulos en esas columnas.
*Pistas: `fillna`, `.median()`, `.mode()[0]`, `drop(columns=...)`.*

In [ ]:
# age → mediana

# embarked → moda

# deck → se elimina la columna

# verificación: nulos restantes


**Responde aquí:** ¿por qué mediana y no media para `age`? ¿Por qué eliminar `deck` completa en
vez de imputarla? ¿Y por qué imputar `embarked` con la moda es poco riesgoso aquí?

> *(tus respuestas)*

### 7. De texto a números: encoding

Los modelos operan sobre números. Antes de codificar, clasifica cada columna —
¿**nominal** (sin orden) u **ordinal** (con orden)?

- `sex` (male / female) → ¿?
- `embarked` (S / C / Q — puerto de embarque) → ¿?
- `class` (First / Second / Third) → ¿?

**Tu tarea:**
1. `sex` → 0/1 con `.map({"male": 0, "female": 1})`. *(Binaria: con solo 2 categorías, un 0/1 no
   inventa ningún orden falso.)*
2. `embarked` y `class` → One-Hot con `pd.get_dummies(titanic, columns=[...], drop_first=True)`,
   guardando el resultado en `titanic_enc`.
3. Imprime las columnas nuevas que aparecieron en `titanic_enc`.

**Responde aquí:** `class` sí tiene un orden natural (First > Second > Third). ¿Habría sido mejor
un encoding ordinal (1/2/3) que One-Hot? ¿Existe ya en el dataset una columna que sea exactamente
eso? No hay una única respuesta correcta — argumenta.

> *(tu respuesta)*

---
## Parte 3 · Todo junto

### 8. El flujo completo, de principio a fin

Todo el lab en una sola secuencia, ahora para **predecir quién sobrevivió**:
datos limpios → split → escalar (regla de oro) → KNN → score honesto.

La selección de columnas viene dada; el resto es exactamente lo que ya hiciste con Wine.

**Tu tarea:** completa el split (test 30%, `random_state=42`), escala respetando la regla de oro,
entrena un KNN e imprime su score sobre test. *(Reutiliza los nombres `X_train`, `X_test`,
`y_train`, `y_test` — la celda de la sección 9 los espera así.)*

In [ ]:
features = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
cols_extra = [c for c in titanic_enc.columns if c.startswith(("embarked_", "class_"))]

X = titanic_enc[features + cols_extra]
y = titanic_enc["survived"]

# 1. split (test_size=0.3, random_state=42)

# 2. escalar: fit_transform en train, transform (sin fit) en test

# 3. KNN + score sobre test


**Responde aquí:** ¿qué score obtuviste? Corre también la versión **sin escalar** (mismo split):
¿cuánto pierde KNN? ¿Se repitió la historia de Wine en un dataset real?

> *(tu respuesta)*

### 9. `Pipeline`: lo mismo, sin poder equivocarse *(si sobra tiempo)*

`sklearn` permite encadenar escalador + modelo en un solo objeto: al hacer `fit` aplica
`fit_transform` internamente solo con train, y al hacer `score` aplica `transform` al test.
Con un pipeline es **imposible** olvidar transformar el test o hacer `fit` donde no corresponde.
Hoy solo se muestra; vuelve más adelante en el curso.

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)   # mismo resultado que tu versión manual

### 10. Ejercicio propuesto (no evaluado)

Repite el flujo de Titanic con un **árbol de decisión** en vez de KNN, con y sin escalar
(mismo split de la sección 8). ¿Cuánto le importó el escalamiento al árbol?
¿Confirma lo que respondiste en la sección 4?

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# árbol sin escalar vs árbol escalado (random_state=42), mismo split de la sección 8


---
**Moraleja del lab:** preprocesar no es un trámite antes del modelo. Hoy el mismo KNN pasó de
mediocre a excelente **sin tocar el algoritmo** — solo cambiando cómo le llegan los datos.

**Anuncios:**
- Próxima clase (26-ago): **selección y extracción de características** — qué hacer cuando hay
  demasiadas columnas, o columnas redundantes o inútiles *(¿notaste alguna hoy?)*; primer vistazo a PCA.
- La **Tarea 1** se enunciará próximamente, condensando las clases 1 a 3.